<a href="https://colab.research.google.com/github/steffiprog/ML/blob/main/7_%D0%A1NN/Parser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install playwright openpyxl

import asyncio
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
import re
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# Playwright async API
from playwright.async_api import async_playwright

# Настройки
BASE_URL = "https://habr.com"
HUB_URL = "https://habr.com/ru/hubs/machine_learning/articles/"
TARGET_ARTICLES = 600
REQUEST_DELAY = 1.0
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"

# Функции
async def get_soup(url):
    """Загружает страницу с помощью Playwright (асинхронно) и возвращает BeautifulSoup объект."""
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch()
            page = await browser.new_page(user_agent=USER_AGENT)
            await page.goto(url, timeout=60000)
            html_content = await page.content()
            await browser.close()
            return BeautifulSoup(html_content, 'html.parser')
    except Exception as e:
        print(f"Ошибка при загрузке {url} с Playwright: {e}")
        return None

async def parse_article(url):
    """Парсит страницу одной статьи (асинхронно) и возвращает словарь с данными."""
    soup = await get_soup(url)
    if not soup:
        return None

    # 1. Заголовок
    title_tag = soup.find('h1', class_='tm-title tm-title_h1')
    if not title_tag:
        title_tag = soup.find('h1', class_='post__title')
    title = title_tag.text.strip() if title_tag else "Без заголовка"

    # 2. Автор
    author_tag = soup.find('a', class_=re.compile('tm-user-info__username|tm-user-card__link'))
    author = author_tag.text.strip() if author_tag else "Не указан"

    # 3. Дата публикации
    time_tag = soup.find('time')
    date = time_tag['datetime'] if time_tag and time_tag.has_attr('datetime') else "Не указана"

    # 4. Уровень сложности
    level = "Не определен"

    # 5. Подтемы (теги/хабы)
    subtopics = []
    hub_links = soup.find_all('a', class_='tm-publication-hub__link')
    for link in hub_links:
        span_tag = link.find('span')
        if span_tag:
            subtopics.append(span_tag.text.strip())

    tag_links = soup.find_all('a', class_='tm-tag__link')
    for link in tag_links:
        subtopics.append(link.text.strip())

    subtopics = list(set(subtopics)) # Исключить дублирование

    # 6. Текст статьи (все параграфы внутри основного контента)
    content_div = soup.find('div', class_='tm-article-presenter__body')
    if not content_div:
        content_div = soup.find('div', class_='article-formatted-body_version-2')
    if not content_div:
        content_div = soup.find('div', class_='article-formatted-body')
    if not content_div:
        content_div = soup.find('div', id='post-content-body')

    text = ""
    if content_div:
        paragraphs = content_div.find_all(['p', 'h2', 'h3', 'ul', 'ol', 'blockquote'])

        parsed_blocks = []
        for block in paragraphs:
            if block.name == 'p':
                parsed_blocks.append(block.get_text(separator=' ', strip=True))
            elif block.name in ['h2', 'h3']:
                parsed_blocks.append(f"\n\n## {block.get_text(strip=True)}\n")
            elif block.name in ['ul', 'ol']:
                list_items = [li.get_text(strip=True) for li in block.find_all('li')]
                parsed_blocks.append('\n'.join([f"- {item}" for item in list_items]))
            elif block.name == 'blockquote':
                parsed_blocks.append(f"> {block.get_text(strip=True)}")
        text = "\n\n".join([pb for pb in parsed_blocks if pb.strip()])
    else:
        text = "Текст не найден"

    return {
        "title": title,
        "author": author,
        "date": date,
        "level": level,
        "subtopics": subtopics,
        "text": text,
        "url": url
    }

async def collect_articles():
    """Основная логика сбора статей со всех страниц пагинации (асинхронно)."""
    all_articles = []
    page = 1
    collected_urls = set()  # Для избежания дубликатов

    print(f"Начинаем сбор до {TARGET_ARTICLES} статей из раздела ML...")

    while len(all_articles) < TARGET_ARTICLES:
        # Формируем URL текущей страницы пагинации
        if page == 1:
            page_url = HUB_URL
        else:
            # Типичная пагинация на Хабре: /pageN/
            page_url = f"https://habr.com/ru/hubs/machine_learning/articles/page{page}/"

        print(f"\nОбработка страницы {page}: {page_url}")
        soup = await get_soup(page_url)

        if not soup:
            print("Не удалось загрузить страницу пагинации. Завершаем.")
            break

        # Находим все ссылки на статьи на текущей странице
        current_page_article_links = set()
        for link_tag in soup.find_all('a', class_=['tm-title__link', 'tm-article-snippet__title-link']):
            href = link_tag.get('href')
            if href:
                full_url = urljoin(BASE_URL, href)
                # Фильтр
                if ('/post/' in full_url or '/articles/' in full_url) and full_url not in collected_urls:
                    current_page_article_links.add(full_url)

        article_links_list = list(current_page_article_links)

        print(f"  Найдено {len(article_links_list)} новых ссылок на статьи на этой странице.")

        # Если ссылок не найдено на этой странице, это может быть конец парсинга.
        if not article_links_list and page > 1:
            print("  На этой странице не найдено новых ссылок на статьи. Завершаем сбор.")
            break

        # Обрабатываем каждую ссылку на этой странице
        for idx, article_url in enumerate(article_links_list):
            if len(all_articles) >= TARGET_ARTICLES:
                break

            print(f"  Парсинг [{len(all_articles)+1}/{TARGET_ARTICLES}]: {article_url}")
            article_data = await parse_article(article_url)

            if article_data:
                all_articles.append(article_data)
                collected_urls.add(article_url)
                print(f"    Заголовок: {article_data.get('title', 'N/A')[:60]}...")
            else:
                print(f"    Не удалось распарсить статью: {article_url}")

            # Пауза между запросами статей
            time.sleep(REQUEST_DELAY)

        # Пауза между страницами
        time.sleep(REQUEST_DELAY)

        page += 1

    print(f"\nСбор завершён. Собрано {len(all_articles)} статей.")
    return all_articles

def save_to_excel(articles, filename="habr_ml_articles.xlsx"):
    """Сохраняет список статей в Excel файл."""
    wb = Workbook()
    ws = wb.active
    ws.title = "Habr ML Articles"

    # Заголовки
    headers = ["Заголовок", "Автор", "Дата", "Уровень", "Подтемы", "Текст статьи", "URL"]
    header_font = Font(name="Arial", bold=True, size=11, color="FFFFFF")
    header_fill = PatternFill("solid", start_color="2E75B6")
    header_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for col_idx, header in enumerate(headers, start=1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment

    # Данные
    body_font = Font(name="Arial", size=10)
    wrap_alignment = Alignment(vertical="top", wrap_text=True)
    alt_fill = PatternFill("solid", start_color="F0F8FF")

    for row_idx, article in enumerate(articles, start=2):
        ws.cell(row=row_idx, column=1, value=article["title"])
        ws.cell(row=row_idx, column=2, value=article["author"])
        ws.cell(row=row_idx, column=3, value=article["date"])
        ws.cell(row=row_idx, column=4, value=article["level"])
        ws.cell(row=row_idx, column=5, value=", ".join(article["subtopics"]))
        ws.cell(row=row_idx, column=6, value=article["text"])
        ws.cell(row=row_idx, column=7, value=article["url"])

        for col_idx in range(1, 8):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.font = body_font
            cell.alignment = wrap_alignment
            if row_idx % 2 == 0:
                cell.fill = alt_fill

        ws.row_dimensions[row_idx].height = 80  # Высота строки для текста

    # Ширина колонок
    ws.column_dimensions["A"].width = 50   # Заголовок
    ws.column_dimensions["B"].width = 20   # Автор
    ws.column_dimensions["C"].width = 15   # Дата
    ws.column_dimensions["D"].width = 15   # Уровень
    ws.column_dimensions["E"].width = 30   # Подтемы
    ws.column_dimensions["F"].width = 100  # Текст
    ws.column_dimensions["G"].width = 60   # URL

    # Закрепить шапку
    ws.freeze_panes = "A2"

    # Сохраняем
    wb.save(filename)
    print(f"Данные сохранены в файл: {filename}")

# Запуск
articles = await collect_articles()
if articles:
    save_to_excel(articles)
    print(f"\nГотово! Собрано {len(articles)} статей.")
else:
    print("Не удалось собрать ни одной статьи. Проверьте интернет или структуру сайта.")

Начинаем сбор до 600 статей из раздела ML...

Обработка страницы 1: https://habr.com/ru/hubs/machine_learning/articles/
  Найдено 20 новых ссылок на статьи на этой странице.
  Парсинг [1/600]: https://habr.com/ru/articles/1045903/
    Заголовок: raFTI: как сопоставлять «хаотичные» названия вин...
  Парсинг [2/600]: https://habr.com/ru/companies/it_sense/articles/1046063/
    Заголовок: Внедрение ИИ-агента глазами QA: полгода от скепсиса до 1600 ...
  Парсинг [3/600]: https://habr.com/ru/companies/ru_mts/articles/1043156/
    Заголовок: Курсы по промптингу? Почитайте детям сказку на ночь...
  Парсинг [4/600]: https://habr.com/ru/companies/otus/articles/1044348/
    Заголовок: Дообучаем FLUX.2 [klein] за час на одной видеокарте: LoRA, D...
  Парсинг [5/600]: https://habr.com/ru/articles/1046081/
    Заголовок: Архиватор рождённый из теории предельного сжатия вселенной...
  Парсинг [6/600]: https://habr.com/ru/companies/just_ai/articles/1045967/
    Заголовок: Промпт-инъекции в реальных д

In [ ]:
# Install Playwright's system dependencies
!playwright install-deps

Installing dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,053 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 MB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:14